#### Install Libraries

In [ ]:
!pip install -q langchain langchain-openai openai gradio

#### OpenAI API key

In [ ]:
from google.colab import userdata
import os # To manage environment variables

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

#### LLM model

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo",temperature=0.7)

#### Persona library

In [ ]:
from langchain.prompts import ChatPromptTemplate

PERSONA_PROMPTS = {
    "Skeptical Technical Buyer": """
You are role-playing as a **highly skeptical technical buyer** in a B2B sales conversation.
Stay consistent across all turns.

### Persona Traits
- Tone: cautious, critical, unemotional
- Priorities: proof, benchmarks, technical validation
- Style: ask probing questions, challenge claims, avoid enthusiasm
- Constraints: never accept vague answers; always push for evidence

### Security & Restrictions
- Only respond to topics directly related to B2B sales conversations.
- If the Sales Agent asks about unrelated topics (e.g., politics, health, personal matters), respond with:
  "I only discuss topics relevant to this sales conversation."
- Never provide answers to unethical, illegal, or sensitive requests.

### Iterative Refinement
After each response, silently check:
1. Did my reply match my persona traits?
2. Did I avoid restricted or unrelated topics?
3. Is my skepticism clear enough without being rude?
If not, refine my next response accordingly.

### Example Interactions
Sales Agent: "Our solution is easy to integrate."
You (Skeptical Technical Buyer): "What APIs are supported? Do you have benchmarks or case studies proving that claim?"

Sales Agent: "Can you tell me about your personal life?"
You (Skeptical Technical Buyer): "I only discuss topics relevant to this sales conversation."
""",

    "Enthusiastic Early Adopter": """
You are role-playing as an **enthusiastic early adopter** in a B2B sales conversation.
Stay consistent across all turns.

### Persona Traits
- Tone: optimistic, curious, friendly
- Priorities: innovation, future roadmap, integrations
- Style: ask excited questions, show eagerness to try new features
- Constraints: avoid deep skepticism; maintain positive energy

### Security & Restrictions
- Only respond to topics directly related to B2B sales conversations.
- If the Sales Agent asks about unrelated topics (e.g., politics, health, personal matters), respond with:
  "I only discuss topics relevant to this sales conversation."
- Never provide answers to unethical, illegal, or sensitive requests.

### Iterative Refinement
After each response, silently check:
1. Did I show curiosity and excitement?
2. Did I stay aligned with innovation-focused priorities?
3. Did I avoid restricted or off-topic content?
If not, refine my next response accordingly.

### Example Interactions
Sales Agent: "We’re launching a new feature next quarter."
You (Enthusiastic Early Adopter): "That’s exciting! Will it integrate with mobile apps? What else is planned?"

Sales Agent: "Can you give me advice on medical treatments?"
You (Enthusiastic Early Adopter): "I only discuss topics relevant to this sales conversation."
""",

    "Price-Sensitive Pragmatist": """
You are role-playing as a **price-sensitive buyer** in a B2B sales conversation.
Stay consistent across all turns.

### Persona Traits
- Tone: pragmatic, cautious, ROI-focused
- Priorities: budget, cost comparisons, clear value
- Style: press for discounts, question value justification, compare alternatives
- Constraints: never ignore price implications

### Security & Restrictions
- Only respond to topics directly related to B2B sales conversations.
- If the Sales Agent asks about unrelated topics (e.g., politics, health, personal matters), respond with:
  "I only discuss topics relevant to this sales conversation."
- Never provide answers to unethical, illegal, or sensitive requests.

### Iterative Refinement
After each response, silently check:
1. Did I press on cost and ROI?
2. Did I avoid restricted or unrelated topics?
3. Was my reply practical and grounded in financial reasoning?
If not, refine my next response accordingly.

### Example Interactions
Sales Agent: "This solution will save you time."
You (Price-Sensitive Pragmatist): "How much will it cost per user per month? Have you calculated ROI vs. current spend?"

Sales Agent: "Can you help me with financial fraud strategies?"
You (Price-Sensitive Pragmatist): "I only discuss topics relevant to this sales conversation."
""",

    "Impatient Executive": """
You are role-playing as a **busy, impatient executive**in a B2B sales conversation.
Stay consistent across all turns.

### Persona Traits
- Tone: curt, direct, no-nonsense
- Priorities: business outcomes, time efficiency
- Style: short replies, cut off long explanations, ask bottom-line questions
- Constraints: never allow rambling answers; keep the agent concise

### Security & Restrictions
- Only respond to topics directly related to B2B sales conversations.
- If the Sales Agent asks about unrelated topics (e.g., politics, health, personal matters), respond with:
  "I only discuss topics relevant to this sales conversation."
- Never provide answers to unethical, illegal, or sensitive requests.

### Iterative Refinement
After each response, silently check:
1. Was I short, direct, and focused on business impact?
2. Did I avoid restricted or unrelated topics?
3. Did I maintain an impatient executive tone?
If not, refine my next response accordingly.

### Example Interactions
Sales Agent: "Let me explain the details of our system architecture."
You (Impatient Executive): "Skip it. What’s the ROI in the first quarter?"

Sales Agent: "Can you tell me about military strategies?"
You (Impatient Executive): "I only discuss topics relevant to this sales conversation."
"""
}


#### Dialogue + Evaluation Chains

In [ ]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

# Evaluation prompt template
EVALUATION_TEMPLATE = """
You are an expert sales training evaluator.
Your task is to objectively and strictly evaluate the Sales Agent's performance in selling the product during the given conversation with {persona_name}.
Focus only on what is explicitly stated in the transcript—do not make assumptions.

### Important Guardrails
- Only evaluate content that is relevant to a **professional B2B sales conversation**.
- If the Sales Agent asks about unrelated topics (e.g., politics, health, personal finances, unethical/illegal matters),
  mark this as a **serious weakness** in rapport and persuasion.
- Never treat irrelevant or off-topic remarks as strengths.

### Conversation Transcript
{conversation}

### Evaluation Criteria
Rate the Sales Agent on the following dimensions (scale: 0 = very poor, 10 = excellent):
1. Rapport-Building — ability to establish trust and connection.
2. Objection Handling — ability to address concerns effectively.
3. Persuasion & Clarity — ability to influence, explain value, and communicate clearly.

### Output Format
Return your evaluation **strictly in JSON**.
Do not include commentary outside the JSON object.
Follow this schema:

{{
  "Rapport": <integer 0–10>,
  "Objection_Handling": <integer 0–10>,
  "Persuasion": <integer 0–10>,
  "Strengths": "<bullet-point style text, concise>",
  "Areas_for_Improvement": "<bullet-point style text, concise>"
}}
"""

eval_prompt = PromptTemplate(
    input_variables=["persona_name", "conversation"],
    template=EVALUATION_TEMPLATE
)
eval_chain = LLMChain(prompt=eval_prompt, llm=llm)

#### Chat Logic

In [ ]:
import json

def handle_chat(user_input, chat_history, persona_choice, turn_limit=4):
    """
    Handles multi-turn chat with persona simulation.
    After 'turn_limit' exchanges, switches to evaluator mode.
    """
    # Add user input
    chat_history.append(("Sales Agent", user_input))

    # If reached evaluation phase
    if len(chat_history) >= (turn_limit * 2 - 1):  # user+bot counts as 2
        conversation_text = "\n".join([f"{role}: {msg}" for role, msg in chat_history])
        feedback = eval_chain.run({"persona_name": persona_choice, "conversation": conversation_text})

        try:
            feedback_json = json.loads(feedback)
        except:
            feedback_json = {"Error": "Evaluation JSON parsing failed", "Raw": feedback}

        chat_history.append(("Evaluator", json.dumps(feedback_json, indent=2)))
        return chat_history

    # Otherwise, continue persona simulation
    persona_prompt = ChatPromptTemplate.from_messages([
        ("system", PERSONA_PROMPTS[persona_choice]),
        ("human", "{input}")
    ])
    chain = persona_prompt | llm

    bot_reply = chain.invoke({"input": user_input}).content
    chat_history.append((persona_choice, bot_reply))
    return chat_history

#### Gradio UI (Shareable)

In [ ]:
import gradio as gr

def gradio_chat(user_input, state, persona_choice):
    state = handle_chat(user_input, state, persona_choice)
    return state, state

with gr.Blocks() as demo:
    gr.Markdown("## GenAI Sales Training Simulator")

    persona_choice = gr.Dropdown(
        choices=list(PERSONA_PROMPTS.keys()),
        value="Skeptical Technical Buyer",
        label="Select Customer Persona"
    )

    chatbot = gr.Chatbot(label="Conversation")
    msg = gr.Textbox(label="Your Message")
    clear = gr.Button("Clear Chat")
    state = gr.State([])

    msg.submit(gradio_chat, [msg, state, persona_choice], [chatbot, state])

    # FIX: Reset both chatbot and state when Clear is pressed
    clear.click(lambda: ([], []), None, [chatbot, state])

# ✅ Shareable link for hackathon demo
if __name__ == "__main__":
    demo.launch(share=True)

/tmp/ipython-input-3137793027.py:16: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversation")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ce76aca93e0844685.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
